# Skill-selection runs on opencode (groups A and B)

Installs each cluster's skills (real name + description, stub body) in a fresh git repo, runs every golden request
`N_TRIALS` times through `opencode run`, and scores which skill the agent loaded. Then reports pass@k / pass^k.

Each finished trial is appended to `results/<group>_<model>.csv`, so re-running the notebook resumes where it stopped.

In [ ]:
import os, csv, json, shutil, subprocess, tempfile, pathlib, threading
from math import comb
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / "data/golden.jsonl").exists(): ROOT = ROOT.parent

# OpenRouter key from .env (never printed)
for line in open(ROOT / ".env"):
    if line.startswith("Open_Router_Key="):
        os.environ["OPENROUTER_API_KEY"] = line.split("=", 1)[1].strip().strip("\"'")

GROUPS = ["A", "B"]
MODELS = {"deepseek": "opencode/deepseek-v4.1-flash-free",
          "qwen": "openrouter/qwen/qwen3-coder-next"}   # qwen is paid: needs Open_Router_Key in .env
N_TRIALS = 3
WORKERS = 6
TIMEOUT = 300   # s per trial
BODY = "Stub skill for an evaluation. Reply with the single line 'done' and do nothing else.\n"
FIELDS = ["id", "cluster", "tier", "trial", "request", "gold", "acceptable", "chosen_first", "all_loaded", "outcome", "error"]

## Data: golden requests and installed skills

In [ ]:
golden = [json.loads(l) for l in open(ROOT / "data/golden.jsonl")]
skills = defaultdict(list)
for r in csv.DictReader(open(ROOT / "data/selected200.csv")):
    skills[r["group"]].append(r)          # keyed by cluster id, e.g. A1, B3

pd.DataFrame([{"group": g, "tasks": sum(t["group"] == g for t in golden),
               "clusters": len({t["cluster_id"] for t in golden if t["group"] == g})} for g in GROUPS])

## Runner

In [ ]:
def workdir(cluster):
    d = pathlib.Path(tempfile.mkdtemp(prefix=f"opencode_{cluster}_"))
    for s in skills[cluster]:
        p = d / ".opencode/skills" / s["name"]
        p.mkdir(parents=True)
        desc = s["description"].replace('"', '\\"').replace("\n", " ")
        (p / "SKILL.md").write_text(f'---\nname: {s["name"]}\ndescription: "{desc}"\n---\n\n{BODY}')
    subprocess.run(["git", "init", "-q"], cwd=d, check=True)
    return d

def run_trial(t, trial, d, model):
    for attempt in range(2):
        # PWD must be set: opencode reads it, not the process cwd, to find the project.
        # own data dir per trial: opencode's shared sqlite db locks under parallel runs
        data = tempfile.mkdtemp(prefix="oc_data_")
        try:
            pr = subprocess.run(["opencode", "run", "-m", model, "--format", "json", t["request"]],
                                capture_output=True, text=True, timeout=TIMEOUT, stdin=subprocess.DEVNULL, cwd=d,
                                env={**os.environ, "PWD": str(d), "XDG_DATA_HOME": data})
            chosen, err_ev = [], ""
            for line in pr.stdout.splitlines():
                try: ev = json.loads(line)
                except ValueError: continue
                p = ev.get("part", {})
                if ev.get("type") == "tool_use" and p.get("tool") == "skill":
                    chosen.append(p["state"]["input"].get("name", ""))
                if ev.get("type") == "error": err_ev = str(ev)[:200]
            err = err_ev or ("" if pr.returncode == 0 else pr.stderr[-200:])
        except subprocess.TimeoutExpired:
            chosen, err = [], "timeout"
        finally:
            shutil.rmtree(data, ignore_errors=True)
        if "database is locked" not in err: break
    first = chosen[0] if chosen else ""
    ok = {*t["gold_names"], *t["acceptable_names"]}
    outcome = "error" if err and not chosen else "correct" if first in ok else "wrong" if first else "no_skill"
    return dict(id=t["id"], cluster=t["cluster_id"], tier=t["tier"], trial=trial, request=t["request"],
                gold="|".join(t["gold_names"]), acceptable="|".join(t["acceptable_names"]),
                chosen_first=first, all_loaded="|".join(dict.fromkeys(chosen)), outcome=outcome, error=err)

def run_group(group, model_key, retry_errors=True):
    """Run all trials for a group; resume from the CSV, optionally rerunning errored trials."""
    out = ROOT / "results" / f"{group}_{model_key}.csv"
    out.parent.mkdir(exist_ok=True)
    done = {}
    if out.exists():
        done = {(r["id"], int(r["trial"])): r for r in csv.DictReader(open(out))}
        if retry_errors: done = {k: v for k, v in done.items() if v["outcome"] != "error"}
    tasks = [t for t in golden if t["group"] == group]
    dirs = {c: workdir(c) for c in {t["cluster_id"] for t in tasks}}
    jobs = [(t, i, dirs[t["cluster_id"]]) for t in tasks for i in range(1, N_TRIALS + 1) if (t["id"], i) not in done]
    print(f"{group}/{model_key}: {len(jobs)} trials to run, {len(done)} already done")
    # rewrite the kept rows, then append each new trial as it finishes
    lock = threading.Lock()
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(done.values()); f.flush()
        def job(j):
            r = run_trial(*j, MODELS[model_key])
            with lock: w.writerow(r); f.flush()
        with ThreadPoolExecutor(WORKERS) as ex: list(ex.map(job, jobs))
    for d in dirs.values(): shutil.rmtree(d, ignore_errors=True)
    return pd.read_csv(out)

## Metrics

In [ ]:
def pass_metrics(df):
    by = df.sort_values("trial").groupby("id")["outcome"].apply(lambda s: [o == "correct" for o in s])
    n = min(len(v) for v in by)
    rows = []
    for k in range(1, n + 1):
        rows.append({"k": k,
                     "pass@k": sum(1 - comb(n - sum(v[:n]), k) / comb(n, k) for v in by) / len(by),
                     "pass^k": sum(comb(sum(v[:n]), k) / comb(n, k) for v in by) / len(by)})
    return pd.DataFrame(rows).round(3).set_index("k")

def summarize(df):
    print(f"{df['id'].nunique()} tasks, {len(df)} trials")
    display(df["outcome"].value_counts().rename("trials").to_frame().T)
    display(pass_metrics(df))
    c = df.assign(ok=df["outcome"] == "correct").groupby("cluster")["ok"].agg(["sum", "count"])
    c.index = pd.Categorical(c.index, sorted(c.index, key=lambda x: (x[0], int(x[1:]))))
    display(c.sort_index().rename(columns={"sum": "correct", "count": "trials"}).T)

## Run

In [ ]:
results = {}
for g in GROUPS:
    for m in MODELS:
        results[(g, m)] = run_group(g, m)

## Results

In [ ]:
for (g, m), df in results.items():
    print(f"\n===== Group {g}  |  {m} =====")
    summarize(df)

### Combined table

In [ ]:
rows = []
for (g, m), df in results.items():
    pm = pass_metrics(df)
    rows.append({"group": g, "model": m, "tasks": df["id"].nunique(), "trials": len(df),
                 **{c: (df["outcome"] == c).sum() for c in ["correct", "wrong", "no_skill", "error"]},
                 **{f"pass@{k}": pm.loc[k, "pass@k"] for k in pm.index},
                 **{f"pass^{k}": pm.loc[k, "pass^k"] for k in pm.index}})
combined = pd.DataFrame(rows)
combined.to_csv(ROOT / "results/summary_AB.csv", index=False)
combined